# 独立 Metropolis-Hastings 算法

> 对应代码：`MathStatsCode/code_in_notes/Chap.MCMC/MCMC_independent_mh.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：用 Mata 实现独立 M-H 算法的通用函数（目标密度、工具密度与抽样函数均作为参数传入）。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.MCMC`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.MCMC")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

**独立的Metropolis-Hastings算法**

本 notebook 用 Mata 语言实现**独立 Metropolis-Hastings 算法的通用抽象**（只定义函数、不执行抽样），展示了把“目标密度、工具密度、抽样函数”作为参数传入的编程技巧。

关键点：
- `mata:` 与 `end` 之间是 Mata 代码块；`MH_independent` 的声明 `real matrix MH_independent(real scalar N_samples, pointer pai, pointer q, pointer q_sampler, real rowvector x0)` 中，`pointer` 类型的参数是**函数指针**——Mata 允许把函数作为参数传递（Python 里传 lambda 的等价物）；`(*pai)(y)` 就是“调用指针指向的函数”；
- 函数内部：`X = J(N_samples, length(x0), .)` 预分配结果矩阵；循环中每次从工具分布抽一个候选点 `y=(*q_sampler)()`；计算接受概率 ρ=min{1, π(y)q(x)/[π(x)q(y)]}（独立 MH：工具分布与当前状态无关）；`u=runiform(1,1)` 后以概率 ρ 接受候选点（接受则 x=y，否则链停在原处）；每轮都把当前状态存入 X 的第 i 行——**被拒绝时重复记录原状态**，这正是马尔可夫链“在某点停留”的体现。

注意 Mata 里 `scalar :+ vector` 要用逐元素运算符 `:+`（普通 `+` 要求维度一致），这是从数学公式直译代码时的常见坑。


In [ ]:
%%stata
clear all
set more off

mata:
// 独立的MCMC算法，输入：
//    N_samples  : 抽样次数
//      pai(x)   : 目标密度函数
//      q(y)     : 工具密度函数
//   q_sampler   : 从q中抽样的函数
//      x0       : 初始值
real matrix MH_independent(real scalar N_samples, pointer pai, pointer q,
                           pointer q_sampler, real rowvector x0)
{
	real matrix X
	real rowvector x, y
	real scalar i, rho, u
	X = J(N_samples, length(x0), .)
	x = x0
	for (i=1; i<=N_samples; i++) {
		y = (*q_sampler)()
		rho = min((1, (*pai)(y)*(*q)(x)/((*pai)(x)*(*q)(y))))
		u = runiform(1, 1)
		if (u <= rho) {
			x = y
		}
		X[i, .] = x
	}
	return(X)
}
end
